# ***Dependencies***

## ***Installation***

In [ ]:
# pytorch 2.4.0, Cuda 12.4.1

%pip install -U transformers==4.48.0
%pip install -U datasets
%pip install -U accelerate
%pip install -U peft
%pip install -U trl          # Latest TRL works with transformers 4.48.0
%pip install -U bitsandbytes

# Logging (must be recent for Pydantic 2.x compatibility)
%pip install -U wandb

# Utility packages
%pip install openpyxl
%pip install scikit-learn
%pip install tiktoken
%pip install protobuf
%pip install sentencepiece

# HuggingFace Hub (newer version required for new transformers)
%pip install "huggingface-hub<1.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.2/69.2 kB 1.6 MB/s eta 0:00:00a 0:00:01
INFO: pip is looking at multiple versions of huggingface-hub to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of huggingface-hub to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Ensure typing_extensions is new enough for pydantic-core
!pip install --no-cache-dir typing_extensions>=4.14.1
# RESTART KERNEL

## ***Imports***

In [ ]:
import os
import yaml
import numpy as np
import pandas as pd

from datasets import Dataset

from dataclasses import asdict, dataclass, field, fields

import torch
import torch.nn as nn
from torch import Tensor
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
from transformers import PreTrainedModel
from transformers import TrainingArguments
from transformers import DataCollatorWithPadding

from trl import SFTTrainer
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

from dotenv import load_dotenv


from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix
)

## ***Download Model***

In [ ]:
!huggingface-cli download medalpaca/medalpaca-7b --local-dir /workspace/models/medalpaca

# ***Configuration and Global Variables***

In [ ]:
root_path = '/workspace'
token = "Your HuggingFace Token"

classification_task_prompts =(
    "You are an expert in cognitive health and language analysis. You will analyze a spoken language transcript from a person describing the 'cookie theft' picture. This is not written text but a transcription of spontaneous speech."
    "\nAnalyze the provided transcript and classify it into one of three categories: 'Healthy' for a healthy cognitive state, 'MC' for a mild cognitive impairment, or 'AD' for Alzheimer's disease."
    "\nProvide only the label ('Healthy', 'MC, or 'AD') as the output. Do not include explanations or additional text."
    "\nText: {text}"
    "\nLabel: "
)

# ***Helper Functions***

In [ ]:
class Metrics:
    @staticmethod
    def compute_metrics(evaluations):
        """
        Computes classification metrics from model outputs.

        Args:
            evaluations (tuple):
                - predictions (np.ndarray): Model logits or probabilities (N x C).
                - labels (np.ndarray): Ground-truth labels (N,).

        Returns:
            dict: {
                'accuracy': float,
                'precision': float (macro),
                'recall': float (macro),
                'f1': float (macro),
                'TP': int|None,
                'FP': int|None,
                'TN': int|None,
                'FN': int|None
            }
            TP/FP/TN/FN are provided only for binary classification.
        """
        predictions, labels = evaluations
        predictions = np.argmax(predictions, axis=1)

        # Core metrics
        acc = accuracy_score(labels, predictions)
        # balanced_acc = balanced_accuracy_score(labels, predictions)
        precision = precision_score(labels, predictions, average='macro', zero_division=0)
        recall = recall_score(labels, predictions, average='macro', zero_division=0)
        f1 = f1_score(labels, predictions, average='macro', zero_division=0)

        # Confusion matrix
        cm = confusion_matrix(labels, predictions)
        if cm.shape == (2, 2):  # Binary classification
            tn, fp, fn, tp = cm.ravel()
        else:
            tn = fp = fn = tp = None  # Multi-class: these don't apply directly

        return {
            'accuracy': acc,
            # 'balanced_accuracy': balanced_acc,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'TP': tp,
            'FP': fp,
            'TN': tn,
            'FN': fn
        }

In [ ]:
class utils:
    @staticmethod
    def softmax(logits):
        """
        Computes row-wise softmax probabilities from logits.

        Args:
            logits (np.ndarray): Array of shape (N, C) containing raw model scores.

        Returns:
            np.ndarray: Softmax probabilities of shape (N, C).
        """
        exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
        return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

    @staticmethod
    def save_results(val_df , test_df , validation_results , test_results, model_name):
        """
        Saves validation and test prediction probabilities to CSV files.

        Args:
            val_df (pd.DataFrame): Validation dataframe with 'text' and 'label'.
            test_df (pd.DataFrame): Test dataframe with 'text' and 'label'.
            validation_results (list[tuple]): Predicted probabilities per validation sample.
            test_results (list[tuple]): Predicted probabilities per test sample.
            model_name (str): Model identifier used in output filenames.

        Returns:
            None
        """
        validation_df_res = pd.DataFrame(
            [(text, gt_label, *vals) for text, gt_label, vals in zip(val_df["text"].to_list(), val_df['label'].to_list(), validation_results)],
            columns=['text', 'gt_label', 'Healthy', 'ADRD'])
        test_df_res = pd.DataFrame([(text, gt_label, *vals) for text, gt_label, vals in
                                    zip(test_df["text"].to_list(), test_df['label'].to_list(), test_results)],
                                   columns=['text', 'gt_label', 'Healthy', 'ADRD'])
        validation_df_res.to_csv(os.path.join(root_path, f"predictions/{model_name}_validation_results.csv"))
        test_df_res.to_csv(os.path.join(root_path, f"predictions/{model_name}_test_results.csv"))

# ***Component 3.2: Classification Head Fine-Tuning***

## ***Configuration***

In [ ]:
@dataclass
class TrainingConfig:
    """
    Configuration container for model fine-tuning and training hyperparameters.
    """
    
    model_name: str = field(#default="mistralai/Mixtral-8x7B-v0.1",
                            metadata={"help": "model address from huggingface or local"})
    
    lora_r: int = field(default=16, metadata={"help": "lora rank"})
    
    lora_dropout: float = field(default=0, metadata={"help": "lora dropout"})
    
    lora_target_modules: list = field(
        default=("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "lm_head"),
        metadata={"help": "which mosules should add lora layers - give attention layer nemes"})
    
    use_4bit: bool = field(default=True, metadata={"help": "quantization to 4 bit with bitsand bytes"})
    
    output_dir: str = field(default="./results", metadata={"help": "Directory to save model outputs and checkpoints"})
    
    num_train_epochs: int = field(default=4, metadata={"help": "Number of training epochs"})
    
    bf16: bool = field(default=False, metadata={"help": "Enable bfloat16 precision"})
    
    per_device_train_batch_size: int = field(default=4,
                                             metadata={"help": "Batch size per GPU/TPU core during training"})
    
    per_device_eval_batch_size: int = field(default=4,
                                            metadata={"help": "Batch size per GPU/TPU core during evaluation"})
    
    gradient_accumulation_steps: int = field(default=2, metadata={
        "help": "Number of steps to accumulate gradients before updating model weights"})
    
    gradient_checkpointing: bool = field(default=True,
                                         metadata={"help": "Enable gradient checkpointing to reduce memory usage"})
    
    learning_rate: float = field(default=2e-4, metadata={"help": "Initial learning rate"})
    
    weight_decay: float = field(default=0.001,
                                metadata={"help": "Weight decay to apply to all layers except bias/LayerNorm weights"})
    
    optimizer: str = field(default="paged_adamw_32bit", metadata={"help": "Optimizer to use"})
    
    lr_scheduler_type: str = field(default="linear", metadata={"help": " Learning rate schedule"})
    
    device_map: str = field(default="cuda:0", metadata={"help": "Maximum sequence length to use"})
    
    logging_steps: int = field(default=10, metadata={"help": "Log every X updates steps"})

def load_config(config_path: str = "/workspace/config.yaml") -> TrainingConfig:
    """
    Loads training configuration from a YAML file and returns a TrainingConfig object.

    Args:
        config_path (str): Path to YAML configuration file.

    Returns:
        TrainingConfig: Initialized configuration object.
    """
    with open(config_path, "r") as file:
        config_dict = yaml.safe_load(file)
    return TrainingConfig(**config_dict)

## ***Data Processor***

In [ ]:
class DataProcessor:
    def __init__(self, tokenizer,
                 train_path="data/train.csv",
                 val_path="data/validation.csv",
                 test_path="data/test.csv",
                 input_column = "text",
                 output_column = "label"):
        """
        Initializes the DataProcessor by loading and processing datasets.

        Args:
            tokenizer: The tokenizer to use for text preprocessing.
            train_path: Path to the training dataset.
            val_path: Path to the validation dataset.
            test_path: Path to the test dataset.
            input_column: name of input column in data
            output_column: name of output column in data
        """
        self.tokenizer = tokenizer
        self.train_path = os.path.join(root_path, train_path)
        self.val_path = os.path.join(root_path, val_path)
        self.test_path = os.path.join(root_path, test_path)

        self.input_column = input_column
        # print(input_column)
        self.output_column = output_column

        self.dataset_train = None
        self.dataset_val = None
        self.dataset_test = None
        
    def load_data(self):
        """
        Loads data from CSV files and selects necessary columns.
        """
        train_df = pd.read_csv(self.train_path)
        # train_df = train_df.loc[train_df.valid == 'Yes'].reset_index(drop=True)
        train_df = train_df[[self.input_column, self.output_column]]
        
        val_df = pd.read_csv(self.val_path)
        # val_df = val_df.loc[val_df.valid == 'Yes'].reset_index(drop=True)
        val_df = val_df[[self.input_column, self.output_column]]
        
        test_df = pd.read_excel(self.test_path)
        # test_df = test_df.loc[test_df.valid == 'Yes'].reset_index(drop=True)
        test_df = test_df[[self.input_column, self.output_column]]

        self.val_df = val_df
        self.test_df = test_df

        self.dataset_train = Dataset.from_pandas(train_df)
        self.dataset_val = Dataset.from_pandas(val_df)
        self.dataset_test = Dataset.from_pandas(test_df)

    def preprocess_record(self, record):
        """
        Tokenizes and processes a single record.

        Args:
            record: A dictionary representing a single dataset entry.
        Returns:
            Processed input dictionary.
        """
        encoded_text = self.tokenizer(
            classification_task_prompts.format(text=record[self.input_column]),
            max_length=600,
            padding="max_length",
            truncation=True
        )
    
        return {
            "input_ids": encoded_text["input_ids"],
            "attention_mask": encoded_text["attention_mask"],
            "labels": record[self.output_column]  # Use the numerical label here
        }
        
    def preprocess_datasets(self):
        """
        Applies preprocessing to datasets using the tokenizer.
        """
        self.dataset_train = self.dataset_train.map(self.preprocess_record)
        self.dataset_train = self.dataset_train.remove_columns([self.input_column, self.output_column])
        
        self.dataset_val = self.dataset_val.map(self.preprocess_record)
        self.dataset_val = self.dataset_val.remove_columns([self.input_column, self.output_column])
        
        self.dataset_test = self.dataset_test.map(self.preprocess_record)
        self.dataset_test = self.dataset_test.remove_columns([self.input_column, self.output_column])

    def format_datasets(self):
        """
        Sets dataset format.
        """

        self.dataset_val.set_format(type='torch')
        self.dataset_test.set_format(type='torch')

    def get_datasets(self):
        """
        Returns datasets (train , validation and test)
        """
        return  self.dataset_train, self.dataset_val, self.dataset_test

    def get_dataloaders(self, batch_size=1):
        """
        Creates DataLoader objects for validation and test datasets.

        Args:
            batch_size (int): Number of samples per batch.

        Returns:
            tuple:
                - validation_loader (DataLoader): Loader for validation dataset.
                - test_loader (DataLoader): Loader for test dataset.
        """
        validation_loader = DataLoader(self.dataset_val, batch_size=batch_size, shuffle=False)
        test_loader = DataLoader(self.dataset_test, batch_size=batch_size, shuffle=False)
        return validation_loader, test_loader


## ***Model***

### ***Base Model***

In [ ]:
class BaseModelLoader:
    """
    A class responsible for loading a tokenizer and model with optional quantization settings.

    Attributes:
        model_name (str): Name of the pretrained model.
        token (str): Authentication token for loading the model (if required).
        use_4bit (bool): Flag to determine if 4-bit quantization should be used.
        bf16 (bool): Flag to determine if bfloat16 should be used instead of float16.
    """

    def __init__(self, args, token: str = ""):
        """
        Initializes the ModelLoader with provided arguments.

        Args:
            args: A namespace or object containing model configuration options.
            token (str, optional): Optional authentication token. Defaults to an empty string.
        """
        self.model_name = args.model_name
        self.token = token
        self.use_4bit = args.use_4bit
        self.bf16 = args.bf16

    def load_tokenizer(self) -> AutoTokenizer:
        """
        Loads the tokenizer for the specified model.

        Returns:
            AutoTokenizer: The loaded tokenizer with padding settings configured.
        """
        tokenizer = AutoTokenizer.from_pretrained(self.model_name, token=self.token)
        tokenizer.pad_token_id = tokenizer.eos_token_id
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "left"
        return tokenizer

    def load_model(self) -> AutoModel:
        """
        Loads the model with or without 4-bit quantization.

        Returns:
            AutoModel: The loaded model configured with the appropriate precision and quantization settings.
        """
        model_kwargs = {
            "pretrained_model_name_or_path": self.model_name,
            "token": self.token,
            "device_map": "auto",
        }

        if self.use_4bit:
            model_kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.bfloat16 if self.bf16 else torch.float16,
            )
        else:
            model_kwargs["torch_dtype"] = torch.bfloat16 if self.bf16 else torch.float16

        return AutoModel.from_pretrained(**model_kwargs)


### ***Apply LoRA Configurations***

In [ ]:
class LoRATrainer:
    def __init__(self, model: PreTrainedModel, args):
        """
        Initializes the LoRATrainer with a model and configuration arguments.
        Args:
            model: The base model to be fine-tuned.
            args: An object containing LoRA-specific configurations.
        """
        self.model = model
        self.args = args
        self.lora_config = None

    def apply_lora(self):
        """
        Configures and applies LoRA adaptation to the model.
        """
        self.lora_config = LoraConfig(
            r=self.args.lora_r,
            lora_alpha=self.args.lora_r * 2,
            target_modules=self.args.lora_target_modules,
            lora_dropout=self.args.lora_dropout,
            bias='none',
            task_type='SEQ_CLS',
            modules_to_save=[]
        )

        self.model = prepare_model_for_kbit_training(self.model)
        self.model = get_peft_model(self.model, self.lora_config)

    def enable_trainable_weights(self):
        """
        Enables training for specific weight and bias parameters in the model.
        """
        # self.model.model.linear.weight.requires_grad = True
        # self.model.model.linear.bias.requires_grad = True
        # self.model.model.linear_fc.weight.requires_grad = True
        # self.model.model.linear_fc.bias.requires_grad = True
        for param in self.model.model.classifier.original_module.parameters():
            param.requires_grad = True

    def get_model(self):
        """
        Returns the modified model.
        """
        return self.model


### ***Classification Head***

In [ ]:
class SequenceClassifier(nn.Module):
    def __init__(self, base_model: PreTrainedModel, num_classes: int = 3):
        """
        A sequence classification model that builds on top of a transformer-based model.

        Args:
            base_model (PreTrainedModel): The pre-trained transformer model.
            num_classes (int, optional): Number of output classes. Defaults to 3.
        """
        super().__init__()
        self.model = base_model
        self.config = base_model.config
        self.num_classes = num_classes

        self.classifier = nn.Sequential(
            nn.Linear(self.config.hidden_size, 512),
            nn.Tanh(),
            nn.Linear(512, 256),
            nn.Tanh(),
            nn.Linear(256, num_classes)
        )

        self.loss_fn = nn.CrossEntropyLoss()

    def forward(
        self,
        input_ids: Tensor,
        attention_mask: Tensor = None,
        position_ids: Tensor = None,
        past_key_values = None,
        inputs_embeds: Tensor = None,
        labels: Tensor = None,
        use_cache: bool = None,
        output_attentions: bool = None,
        output_hidden_states: bool = None,
        return_dict: bool = None,
    ):
        """
        Forward pass through the model.

        Args:
            input_ids (Tensor): Tokenized input tensor.
            attention_mask (Tensor, optional): Attention mask tensor.
            position_ids (Tensor, optional): Position IDs tensor.
            past_key_values (optional): Past key values for transformer caching.
            inputs_embeds (Tensor, optional): Embedded inputs tensor.
            labels (Tensor, optional): Target labels for computing loss.
            use_cache (bool, optional): Whether to use cache.
            output_attentions (bool, optional): Whether to output attentions.
            output_hidden_states (bool, optional): Whether to output hidden states.
            return_dict (bool, optional): Whether to return as dict.

        Returns:
            tuple: (loss, logits) if labels are provided, else logits.
        """
        outputs = self.model(
            input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values,
            inputs_embeds=inputs_embeds,
            use_cache=use_cache,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        hidden_states = outputs.last_hidden_state[:, -1, :]
        logits = self.classifier(hidden_states)
        # print(labels)
        # print(logits)
        loss = self.loss_fn(logits, labels) if labels is not None else None
        return (loss, logits) if loss is not None else logits


## ***Trainer***

In [ ]:
class ModelTrainer:
    def __init__(self, model, dataset_train, dataset_val, tokenizer, collate_fn,compute_metrics , args ):
        """
        Initializes the ModelTrainer with necessary configurations and datasets.
        Args:
            model: The model to be fine-tuned.
            dataset_train: Training dataset.
            dataset_val: Validation dataset.
            tokenizer: Tokenizer for processing input text.
            collate_fn: Function to collate data samples into a batch.
            compute_metrics: Function to compute evaluation metrics.
            args: args for training
        """
        self.model = model
        self.dataset_train = dataset_train
        self.dataset_val = dataset_val
        self.tokenizer = tokenizer
        self.collate_fn = collate_fn
        self.compute_metrics = compute_metrics
        self.args = args
        self.training_args = None
        self.trainer = None

    def setup_training_arguments(self):
        """
        Configures training arguments for the Trainer.
        """
        self.training_args = TrainingArguments(
            output_dir=self.args.output_dir,
            learning_rate=self.args.learning_rate,
            per_device_train_batch_size=self.args.per_device_train_batch_size,
            gradient_accumulation_steps=self.args.gradient_accumulation_steps,
            per_device_eval_batch_size=self.args.per_device_eval_batch_size,
            num_train_epochs=self.args.num_train_epochs,
            logging_steps=self.args.logging_steps,
            # weight_decay=self.args.weight_decay,
            lr_scheduler_type='cosine',
            warmup_ratio=0.03,
            eval_strategy="epoch",
            save_strategy="epoch",
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            save_total_limit=1,
            load_best_model_at_end=True,
            report_to="none",
            optim=self.args.optimizer,
            remove_unused_columns=False
        )

    def setup_trainer(self):
        """
        Initializes the SFTTrainer with model, datasets, tokenizer, and arguments.
        """
        if not self.training_args:
            raise ValueError("Training arguments are not set. Call setup_training_arguments() first.")

        self.trainer = SFTTrainer(
            model=self.model,
            args=self.training_args,
            train_dataset=self.dataset_train,
            eval_dataset=self.dataset_val,
            tokenizer=self.tokenizer,
            data_collator=self.collate_fn,
            compute_metrics=self.compute_metrics,
            packing=True,
        )

    def train(self):
        """
        Starts training the model.
        Returns:
            Training results.
        """
        if not self.trainer:
            raise ValueError("Trainer is not initialized. Call setup_trainer() first.")

        self.trainer.train()

    def predict(self , dataset_val):
        """
        Runs model prediction on a dataset and returns softmax probabilities.

        Args:
            dataset_val: HuggingFace dataset used for evaluation/prediction.

        Returns:
            np.ndarray: Softmax probabilities of shape (N, C).
        """
        validation_results = self.trainer.predict(dataset_val)
        validation_results = validation_results.predictions
        validation_results = utils.softmax(validation_results)
        return validation_results


## ***Model Fine-Tuning and Inference***

You can train all models following the same procedure. Just download model weights and change paths accordingly. 

In [ ]:
load_dotenv()

args = load_config()
base_model_loader = BaseModelLoader(args , token)
base_model = base_model_loader.load_model()
tokenizer = base_model_loader.load_tokenizer()

dp = DataProcessor(tokenizer)
dp.load_data()
dp.preprocess_datasets()
dp.format_datasets()
dataset_train , dataset_val , dataset_test = dp.get_datasets()

model = SequenceClassifier(base_model)
lr_trainer = LoRATrainer(model , args)
lr_trainer.apply_lora()
lr_trainer.enable_trainable_weights()
model = lr_trainer.get_model()

collate_fn = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = ModelTrainer(model, dataset_train, dataset_val, tokenizer, collate_fn, Metrics.compute_metrics , args )
trainer.setup_training_arguments()
trainer.setup_trainer()
trainer.train()

val_predictions = trainer.predict(dataset_val)
test_predictions = trainer.predict(dataset_test)

utils.save_results(dp.val_df , dp.test_df , val_predictions , test_predictions, 'llama8B_3layers')